# YOLO11s 베이스라인 학습

**전제:** 팀원 노트북(`pill_detection_dataset.ipynb`)을 실행해 `data/processed` 아래에
`images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml` 이 생성돼 있어야 합니다.
이 노트북(`notebooks/` 안에 위치)은 그 `data.yaml`만 받아 YOLO11s를 학습합니다.

베이스라인은 팀 합의대로 **기본값 위주**로 돌립니다.


## 1. 설치 환경 재현성

In [1]:
# ============================================================
# 1. 설치, 환경, 재현성
# ============================================================

# Colab이면 실행 (로컬은 최초 1회만)
!pip -q install ultralytics

import os, random, numpy as np, torch
from pathlib import Path
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch: 2.11.0+cu128 | CUDA: True | device: 0


## 2. 경로 설정

In [ ]:
# ============================================================
# 2-1. 구글 드라이브 마운트
# ============================================================
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

In [2]:
# ============================================================
# 2-2. 경로 설정 (모든 경로를 여기서 한 번에 관리)
# ============================================================
PROJECT_ROOT = Path("/content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection")

DATASET_DIR     = PROJECT_ROOT / "data" / "processed"          # YOLO 데이터셋 (images/labels/data.yaml)
TEST_IMAGE_DIR  = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data"
                   / "sprint_ai_project1_data_260809_baseline_dataset" / "test_images")   # Kaggle 테스트 842장
ANNOTATION_DIR = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data"
                  / "sprint_ai_project1_data_260809_baseline_dataset" / "train_annotations")
WEIGHTS_PATH    = PROJECT_ROOT / "outputs" / "yolo" / "yolo11s_baseline_noaug" / "weights" / "best.pt"
SUBMISSION_PATH = PROJECT_ROOT / "outputs" / "submissions" / "submission.csv"

# 존재 확인
print("DATASET_DIR    exists:", DATASET_DIR.exists(), "(필수)")
print("TEST_IMAGE_DIR exists:", TEST_IMAGE_DIR.exists(), "(필수)")
print("ANNOTATION_DIR exists:", ANNOTATION_DIR.exists(), "(필수)")
print("WEIGHTS_PATH   exists:", WEIGHTS_PATH.exists(), "(학습 후 생성)")
print("SUBMISSION dir exists:", SUBMISSION_PATH.parent.exists(), "(제출 시 생성)")

Mounted at /content/drive
DATASET_DIR    exists: True
TEST_IMAGE_DIR exists: True
WEIGHTS_PATH   exists: True
SUBMISSION dir exists: True


## 3. data.yaml

In [4]:
# ==================================================================
# 3. data.yaml 경로 이식성 처리
# ==================================================================

import yaml
src_yaml = DATASET_DIR / "data.yaml"
cfg = yaml.safe_load(open(src_yaml, encoding="utf-8"))
cfg["path"] = str(DATASET_DIR.resolve())
yaml.safe_dump(cfg, open(src_yaml, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
RUNTIME_YAML = src_yaml          # 이후 코드는 그대로 RUNTIME_YAML 사용

assert (DATASET_DIR / "images" / "train").is_dir(), \
    "images/train 이 없습니다. pill_detection_dataset.ipynb 를 먼저 실행해 data/processed 를 만드세요."

print("클래스 수:", cfg["nc"], "| yaml:", RUNTIME_YAML)

클래스 수: 56 | yaml: /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/data/processed/data.yaml


## 4. 학습 (W&B 로깅 포함)

In [8]:
# ==================================================================
# 4-1. W&B 설치 및 로그인
# ==================================================================
!pip -q install wandb
import wandb
from ultralytics import settings

settings.update({"wandb": True})          # Ultralytics 내장 W&B 로깅 ON
wandb.login()   # 최초 1회, wandb.ai/authorize 의 API 키

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [9]:
# ============================================================
# 4-2. 베이스라인 학습 (YOLO11s) + W&B 로깅
# ============================================================

wandb.init(project="pill-object-detection", name="yolo11s-baseline-noaug",
           entity="diokim17",                       # ← 팀 entity (조직 -org 아님)
           job_type="train", tags=["baseline","yolo11s","noaug"])

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(RUNTIME_YAML), epochs=100, imgsz=640, batch=16, # 메모리 부족하면 batch 8 또는 -1(자동)
    seed=SEED, deterministic=True, device=DEVICE,
    project=str(PROJECT_ROOT / "outputs/yolo"),
    name="yolo11s_baseline_noaug", exist_ok=True,  #_noag 는 증강없이 라는 뜻
    # ── 기본 augmentation 모두 끄기 ──
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
)
print("결과 폴더:", results.save_dir)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/data/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosa

wandb: WARNING Tried to log to step 100 that is less than the current step 102. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


lr/pg0,█████▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁
lr/pg1,▆███▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁
lr/pg2,▃████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▃▂▂▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▄██████████████████████████████████████
metrics/mAP50-95(B),▁▂▄▆▇███████████████████████████████████
metrics/precision(B),▁▃▆▇▇█▇▇█▇▇▇▇▇█▇▇▇████▇██▇█▇████████████
metrics/recall(B),▁▃▇▆▇▇▇▇███▆█▇▇▇███▇▇▇▇██▇▇▇▇███████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


결과 폴더: /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/outputs/yolo/yolo11s_baseline_noaug


## 5. 검증 (mAP)


In [11]:
# ==================================================================
# 5. 검증 지표 (mAP)
# ==================================================================
m = model.val(data=str(RUNTIME_YAML), split="val", device=DEVICE)
print(f"최종 Best Validation mAP@0.5:0.95: {m.box.map:.4f}")
print(f"          mAP@0.5 : {m.box.map50:.4f}")
print(f"          mAP@0.75: {m.box.map75:.4f}")

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 486.6±25.1 MB/s, size: 1728.8 KB)
val: Scanning /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/data/processed/labels/val.cache... 16 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16/16 3.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s
                   all         16         51      0.898      0.896      0.992      0.991
           일양하이트린정 2mg         13         13      0.971          1      0.995      0.995
    기넥신에프정(은행엽엑스)(수출용)          3          3      0.922          1      0.995      0.995
          뉴로메드정(옥시라세탐)          4          4      0.918       0.75      0.945      0.945
     에빅사정(메만틴염산염)(비매품)          1          1      0.828          1      0.995      0.995
             리피토정 20mg          3          3          1      0.386    

## 6. 예측 시각화

In [12]:
# ==================================================================
# 6. 예측 시각화 (검증 이미지)
# ==================================================================

val_imgs = sorted((DATASET_DIR / "images" / "val").glob("*.png"))[:6]
pred = model.predict(
    val_imgs, imgsz=640, conf=0.25, max_det=4, device=DEVICE,
    save=True, project=str(PROJECT_ROOT / "outputs/yolo"), name="pred_val", exist_ok=True,
)
print("시각화 저장 위치:", pred[0].save_dir)


0: 640x512 1 일양하이트린정 2mg, 1 오마코연질캡슐(오메가-3-산에틸에스테르90), 1 아토젯정 10/40mg, 9.4ms
1: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 9.4ms
2: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 9.4ms
3: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 9.4ms
4: 640x512 1 일양하이트린정 2mg, 1 에빅사정(메만틴염산염)(비매품), 1 플라빅스정 75mg, 9.4ms
5: 640x512 1 일양하이트린정 2mg, 1 종근당글리아티린연질캡슐(콜린알포세레이트) , 1 플라빅스정 75mg, 9.4ms
Speed: 2.1ms preprocess, 9.4ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 512)
Results saved to /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/outputs/yolo/pred_val
시각화 저장 위치: /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/outputs/yolo/pred_val


## 7. 역매핑

In [3]:
# ============================================================
# 7. YOLO 클래스(0~55) → 원본 category_id 역매핑
# ============================================================
import json

def build_yolo_to_category_id(annotation_dir: Path) -> dict:
    category_ids = set()
    for json_path in annotation_dir.rglob("*.json"):
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        for cat in data.get("categories", []):
            category_ids.add(int(cat["id"]))
    # pill_detection 파이프라인과 동일하게: 정렬 후 0부터 인덱싱
    sorted_ids = sorted(category_ids)
    return {yolo_id: cid for yolo_id, cid in enumerate(sorted_ids)}

yolo2cat = build_yolo_to_category_id(ANNOTATION_DIR)
assert len(yolo2cat) == 56, f"클래스 수 이상: {len(yolo2cat)}개"

def to_category_id(yolo_cls: int) -> int:
    return yolo2cat[yolo_cls]      # 예: 3 → 58873

print("클래스 수:", len(yolo2cat))
print("앞 5개 (yolo_id → category_id):", list(yolo2cat.items())[:5])

클래스 수: 56
앞 5개 (yolo_id → category_id): [(0, 1900), (1, 2483), (2, 3351), (3, 3483), (4, 3544)]


## 8. 제출

In [13]:
# ============================================================
# 8. Kaggle 제출 파일 생성
# ============================================================
import csv, gc
from PIL import Image

model = YOLO(str(WEIGHTS_PATH))          # best.pt 로드 (세션 재시작 후 추론만도 가능)

test_files = sorted(TEST_IMAGE_DIR.glob("*.png"), key=lambda p: int(p.stem))
print("테스트 이미지:", len(test_files), "| 첫 장 크기:", Image.open(test_files[0]).size)

# 제출 전 1장 sanity 체크
r0 = model.predict(str(test_files[0]), imgsz=640, conf=0.25, max_det=4, device=DEVICE, verbose=False)[0]
print("첫 이미지:", test_files[0].name, "| 검출 수:", len(r0.boxes),
      "| category_id 예:", [to_category_id(int(c)) for c in r0.boxes.cls.cpu().numpy()])

테스트 이미지: 842 | 첫 장 크기: (976, 1280)
첫 이미지: 1.png | 검출 수: 2 | category_id 예: [1900, 27926]


In [14]:
# ============================================================
# 8. Kaggle 제출 파일 생성 - continued
# ============================================================

rows, ann_id = [], 1
for i, f in enumerate(test_files):
    r = model.predict(str(f), imgsz=640, conf=0.25, max_det=4, device=DEVICE, verbose=False)[0]
    image_id = int(f.stem)
    b = r.boxes
    for xyxy, conf, cls in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
        x1, y1, x2, y2 = xyxy
        rows.append([ann_id, image_id, to_category_id(int(cls)),
                     int(round(x1)), int(round(y1)),
                     int(round(x2 - x1)), int(round(y2 - y1)), round(float(conf), 4)])
        ann_id += 1
    del r
    if i % 100 == 0:
        gc.collect(); print(f"{i}/{len(test_files)} 처리 중...")

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as file:
    w = csv.writer(file)
    w.writerow(["annotation_id","image_id","category_id","bbox_x","bbox_y","bbox_w","bbox_h","score"])
    w.writerows(rows)
print("작성 완료:", SUBMISSION_PATH, "| 행:", len(rows))

0/842 처리 중...
100/842 처리 중...
200/842 처리 중...
300/842 처리 중...
400/842 처리 중...
500/842 처리 중...
600/842 처리 중...
700/842 처리 중...
800/842 처리 중...
작성 완료: /content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/outputs/submissions/submission.csv | 행: 2852


In [15]:
# ============================================================
# 8. 저장한 Kaggle 제출 파일을 다시 읽어 확인
# ============================================================
import pandas as pd
df = pd.read_csv(SUBMISSION_PATH)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("annotation_id 고유:", df["annotation_id"].is_unique)
print("image_id 수:", df["image_id"].nunique())                       # 검출된 이미지 수
print("category_id 범위:", df["category_id"].min(), "~", df["category_id"].max())
print("category_id 예시:", sorted(df["category_id"].unique())[:10])   # 5자리 코드여야 함
print("score 범위:", round(df["score"].min(),4), "~", round(df["score"].max(),4))

shape: (2852, 8)
columns: ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']
annotation_id 고유: True
image_id 수: 842
category_id 범위: 1900 ~ 41768
category_id 예시: [np.int64(1900), np.int64(2483), np.int64(3351), np.int64(3483), np.int64(3544), np.int64(3743), np.int64(3832), np.int64(4543), np.int64(12081), np.int64(12778)]
score 범위: 0.25 ~ 0.9996
